## 1. Importar librerías necesarias

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output

physical_devices = tf.config.list_physical_devices('GPU')
if physical_devices:
    tf.config.experimental.set_memory_growth(physical_devices[0], True)
    print(f'GPU disponible: {physical_devices[0].name}')
else:
    print('Usando CPU')

## 2. GANs

Una GAN consiste en dos redes neuronales que compiten entre sí:

- Generador (G): transforma ruido aleatorio en datos sintéticos.
- Discriminador (D): estima si una muestra es real o generada.

En este cuaderno las arquitecturas de los ejemplos 1 y 2 son convolucionales tanto para el generador como para el discriminador.

---

# Ejemplo 1: GAN convolucional en MNIST

Usamos un generador con convoluciones transpuestas y un discriminador convolucional.

In [ ]:
(x_train, _), _ = keras.datasets.mnist.load_data()
x_train = (x_train.astype('float32') - 127.5) / 127.5
x_train = np.expand_dims(x_train, axis=-1)

batch_size = 128
latent_dim = 100
train_dataset = tf.data.Dataset.from_tensor_slices(x_train)
train_dataset = train_dataset.shuffle(buffer_size=60000).batch(batch_size).prefetch(tf.data.AUTOTUNE)

print(f'Total de imágenes de entrenamiento: {len(x_train)}')
print(f'Forma de las imágenes: {x_train.shape}')
print(f'Tamaño de batch: {batch_size}')

## Generador convolucional

Partimos de un vector latente, lo proyectamos a un mapa 7x7 y luego lo ampliamos hasta 28x28 con convoluciones transpuestas.

In [ ]:
def build_generator(latent_dim=100):
    model = keras.Sequential([
        layers.Input(shape=(latent_dim,)),
        layers.Dense(7 * 7 * 256, use_bias=False),
        layers.Reshape((7, 7, 256)),
        layers.BatchNormalization(),
        layers.LeakyReLU(0.2),
        layers.Conv2DTranspose(128, kernel_size=4, strides=2, padding='same', use_bias=False),
        layers.BatchNormalization(),
        layers.LeakyReLU(0.2),
        layers.Conv2DTranspose(64, kernel_size=4, strides=2, padding='same', use_bias=False),
        layers.BatchNormalization(),
        layers.LeakyReLU(0.2),
        layers.Conv2D(1, kernel_size=3, padding='same', activation='tanh')
    ], name='generator')
    return model

generator = build_generator(latent_dim)
generator.summary()

## Discriminador convolucional

La imagen de entrada se reduce progresivamente mediante convoluciones con stride antes de clasificarla como real o falsa.

In [ ]:
def build_discriminator():
    model = keras.Sequential([
        layers.Input(shape=(28, 28, 1)),
        layers.Conv2D(64, kernel_size=4, strides=2, padding='same'),
        layers.LeakyReLU(0.2),
        layers.Dropout(0.3),
        layers.Conv2D(128, kernel_size=4, strides=2, padding='same'),
        layers.LeakyReLU(0.2),
        layers.Dropout(0.3),
        layers.Conv2D(256, kernel_size=4, strides=2, padding='same'),
        layers.LeakyReLU(0.2),
        layers.Dropout(0.3),
        layers.Flatten(),
        layers.Dense(1, activation='sigmoid')
    ], name='discriminator')
    return model

discriminator = build_discriminator()
discriminator.summary()

## Configuración de entrenamiento

In [ ]:
cross_entropy = keras.losses.BinaryCrossentropy()
lr = 0.0002
optimizer_G = keras.optimizers.Adam(lr, beta_1=0.5)
optimizer_D = keras.optimizers.Adam(lr, beta_1=0.5)
num_epochs = 50
sample_interval = 5
fixed_noise = tf.random.normal([64, latent_dim])
d_losses = []
g_losses = []

print('Configuración completada.')

## Entrenamiento de la GAN

In [ ]:
@tf.function
def train_step(real_images):
    batch_size = tf.shape(real_images)[0]
    real_labels = tf.ones((batch_size, 1))
    fake_labels = tf.zeros((batch_size, 1))

    with tf.GradientTape() as tape_d:
        real_validity = discriminator(real_images, training=True)
        d_loss_real = cross_entropy(real_labels, real_validity)

        z = tf.random.normal([batch_size, latent_dim])
        fake_images = generator(z, training=True)
        fake_validity = discriminator(fake_images, training=True)
        d_loss_fake = cross_entropy(fake_labels, fake_validity)
        d_loss = d_loss_real + d_loss_fake

    grads_d = tape_d.gradient(d_loss, discriminator.trainable_variables)
    optimizer_D.apply_gradients(zip(grads_d, discriminator.trainable_variables))

    with tf.GradientTape() as tape_g:
        z = tf.random.normal([batch_size, latent_dim])
        gen_images = generator(z, training=True)
        validity = discriminator(gen_images, training=True)
        g_loss = cross_entropy(real_labels, validity)

    grads_g = tape_g.gradient(g_loss, generator.trainable_variables)
    optimizer_G.apply_gradients(zip(grads_g, generator.trainable_variables))

    return d_loss, g_loss

def train_gan():
    for epoch in range(num_epochs):
        epoch_d_loss = 0.0
        epoch_g_loss = 0.0
        num_batches = 0

        for real_imgs in train_dataset:
            d_loss, g_loss = train_step(real_imgs)
            epoch_d_loss += d_loss
            epoch_g_loss += g_loss
            num_batches += 1

        avg_d_loss = epoch_d_loss / num_batches
        avg_g_loss = epoch_g_loss / num_batches
        d_losses.append(float(avg_d_loss))
        g_losses.append(float(avg_g_loss))
        print(f'Epoch [{epoch + 1}/{num_epochs}] | D Loss: {avg_d_loss:.4f} | G Loss: {avg_g_loss:.4f}')

        if (epoch + 1) % sample_interval == 0:
            samples = generator(fixed_noise, training=False).numpy()
            fig, axes = plt.subplots(4, 8, figsize=(12, 6))
            for idx, ax in enumerate(axes.flat):
                if idx < 32:
                    ax.imshow(samples[idx, :, :, 0], cmap='gray')
                    ax.axis('off')
            plt.suptitle(f'Epoch {epoch + 1}')
            plt.tight_layout()
            plt.show()

train_gan()

## Evolución de la función de coste

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(d_losses, label='Discriminator Loss', alpha=0.7)
plt.plot(g_losses, label='Generator Loss', alpha=0.7)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Costes durante el entrenamiento - GAN convolucional')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Muestras generadas

In [ ]:
z = tf.random.normal([64, latent_dim])
generated_imgs = generator(z, training=False).numpy()

fig, axes = plt.subplots(8, 8, figsize=(12, 12))
for idx, ax in enumerate(axes.flat):
    ax.imshow(generated_imgs[idx, :, :, 0], cmap='gray')
    ax.axis('off')
plt.suptitle('Dígitos generados por la GAN convolucional', fontsize=16)
plt.tight_layout()
plt.show()

---

# Ejemplo 2: Conditional GAN convolucional en Fashion-MNIST

La información de clase se integra espacialmente para que el discriminador y el generador trabajen con tensores 2D en lugar de bloques densos puros.

In [ ]:
(x_fashion, y_fashion), _ = keras.datasets.fashion_mnist.load_data()
x_fashion = (x_fashion.astype('float32') - 127.5) / 127.5
x_fashion = np.expand_dims(x_fashion, axis=-1)
fashion_dataset = tf.data.Dataset.from_tensor_slices((x_fashion, y_fashion))
fashion_dataset = fashion_dataset.shuffle(buffer_size=60000).batch(batch_size).prefetch(tf.data.AUTOTUNE)

class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']
num_classes = 10
print(f'Número de clases: {num_classes}')

## Generador condicional convolucional

La etiqueta se embebe y se concatena con el vector latente antes de proyectarse a un mapa espacial.

In [ ]:
def build_conditional_generator(latent_dim=100, num_classes=10, embedding_dim=50):
    noise_input = layers.Input(shape=(latent_dim,))
    label_input = layers.Input(shape=(1,), dtype='int32')

    label_embedding = layers.Embedding(num_classes, embedding_dim)(label_input)
    label_embedding = layers.Flatten()(label_embedding)
    gen_input = layers.Concatenate()([noise_input, label_embedding])

    x = layers.Dense(7 * 7 * 256, use_bias=False)(gen_input)
    x = layers.Reshape((7, 7, 256))(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(0.2)(x)
    x = layers.Conv2DTranspose(128, kernel_size=4, strides=2, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(0.2)(x)
    x = layers.Conv2DTranspose(64, kernel_size=4, strides=2, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(0.2)(x)
    img_output = layers.Conv2D(1, kernel_size=3, padding='same', activation='tanh')(x)

    return keras.Model([noise_input, label_input], img_output, name='conditional_generator')

cond_generator = build_conditional_generator(latent_dim, num_classes)
cond_generator.summary()

## Discriminador condicional convolucional

La etiqueta se convierte en un mapa 28x28 y se concatena como un canal adicional junto a la imagen.

In [ ]:
def build_conditional_discriminator(num_classes=10):
    img_input = layers.Input(shape=(28, 28, 1))
    label_input = layers.Input(shape=(1,), dtype='int32')

    label_embedding = layers.Embedding(num_classes, 28 * 28)(label_input)
    label_embedding = layers.Flatten()(label_embedding)
    label_map = layers.Reshape((28, 28, 1))(label_embedding)

    x = layers.Concatenate(axis=-1)([img_input, label_map])
    x = layers.Conv2D(64, kernel_size=4, strides=2, padding='same')(x)
    x = layers.LeakyReLU(0.2)(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Conv2D(128, kernel_size=4, strides=2, padding='same')(x)
    x = layers.LeakyReLU(0.2)(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Conv2D(256, kernel_size=4, strides=2, padding='same')(x)
    x = layers.LeakyReLU(0.2)(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Flatten()(x)
    validity = layers.Dense(1, activation='sigmoid')(x)

    return keras.Model([img_input, label_input], validity, name='conditional_discriminator')

cond_discriminator = build_conditional_discriminator(num_classes)
cond_discriminator.summary()

## Configuración de la cGAN

In [ ]:
optimizer_cG = keras.optimizers.Adam(lr, beta_1=0.5)
optimizer_cD = keras.optimizers.Adam(lr, beta_1=0.5)
num_epochs_cgan = 50
cd_losses = []
cg_losses = []
fixed_labels = tf.constant([[i % num_classes] for i in range(100)], dtype=tf.int32)
fixed_noise_cgan = tf.random.normal([100, latent_dim])

print('Configuración completada para cGAN.')

## Entrenamiento de la cGAN

In [ ]:
@tf.function
def train_step_conditional(real_images, labels):
    batch_size = tf.shape(real_images)[0]
    labels = tf.expand_dims(labels, axis=-1)
    real_labels_loss = tf.ones((batch_size, 1))
    fake_labels_loss = tf.zeros((batch_size, 1))

    with tf.GradientTape() as tape_cd:
        real_validity = cond_discriminator([real_images, labels], training=True)
        cd_loss_real = cross_entropy(real_labels_loss, real_validity)

        z = tf.random.normal([batch_size, latent_dim])
        fake_labels = tf.random.uniform([batch_size, 1], minval=0, maxval=num_classes, dtype=tf.int32)
        fake_images = cond_generator([z, fake_labels], training=True)
        fake_validity = cond_discriminator([fake_images, fake_labels], training=True)
        cd_loss_fake = cross_entropy(fake_labels_loss, fake_validity)
        cd_loss = cd_loss_real + cd_loss_fake

    grads_cd = tape_cd.gradient(cd_loss, cond_discriminator.trainable_variables)
    optimizer_cD.apply_gradients(zip(grads_cd, cond_discriminator.trainable_variables))

    with tf.GradientTape() as tape_cg:
        z = tf.random.normal([batch_size, latent_dim])
        gen_labels = tf.random.uniform([batch_size, 1], minval=0, maxval=num_classes, dtype=tf.int32)
        gen_images = cond_generator([z, gen_labels], training=True)
        validity = cond_discriminator([gen_images, gen_labels], training=True)
        cg_loss = cross_entropy(real_labels_loss, validity)

    grads_cg = tape_cg.gradient(cg_loss, cond_generator.trainable_variables)
    optimizer_cG.apply_gradients(zip(grads_cg, cond_generator.trainable_variables))

    return cd_loss, cg_loss

def train_conditional_gan():
    for epoch in range(num_epochs_cgan):
        epoch_cd_loss = 0.0
        epoch_cg_loss = 0.0
        num_batches = 0

        for real_imgs, labels in fashion_dataset:
            cd_loss, cg_loss = train_step_conditional(real_imgs, labels)
            epoch_cd_loss += cd_loss
            epoch_cg_loss += cg_loss
            num_batches += 1

        avg_cd_loss = epoch_cd_loss / num_batches
        avg_cg_loss = epoch_cg_loss / num_batches
        cd_losses.append(float(avg_cd_loss))
        cg_losses.append(float(avg_cg_loss))
        print(f'Epoch [{epoch + 1}/{num_epochs_cgan}] | D Loss: {avg_cd_loss:.4f} | G Loss: {avg_cg_loss:.4f}')

        if (epoch + 1) % sample_interval == 0:
            samples = cond_generator([fixed_noise_cgan, fixed_labels], training=False).numpy()
            fig, axes = plt.subplots(10, 10, figsize=(15, 15))
            for idx, ax in enumerate(axes.flat):
                ax.imshow(samples[idx, :, :, 0], cmap='gray')
                ax.set_title(class_names[int(fixed_labels[idx, 0])], fontsize=8)
                ax.axis('off')
            plt.suptitle(f'Generación condicional - Epoch {epoch + 1}', fontsize=16)
            plt.tight_layout()
            plt.show()

train_conditional_gan()

## Evolución de la función de coste

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(cd_losses, label='Conditional Discriminator Loss', alpha=0.7)
plt.plot(cg_losses, label='Conditional Generator Loss', alpha=0.7)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Costes durante el entrenamiento - Conditional GAN convolucional')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Generación controlada por clase

In [ ]:
samples_per_class = 10
total_samples = num_classes * samples_per_class
labels_to_generate = tf.constant([[i] for i in range(num_classes) for _ in range(samples_per_class)], dtype=tf.int32)
z = tf.random.normal([total_samples, latent_dim])
generated_samples = cond_generator([z, labels_to_generate], training=False).numpy()

fig, axes = plt.subplots(num_classes, samples_per_class, figsize=(15, 18))
for class_idx in range(num_classes):
    for sample_idx in range(samples_per_class):
        idx = class_idx * samples_per_class + sample_idx
        ax = axes[class_idx, sample_idx]
        ax.imshow(generated_samples[idx, :, :, 0], cmap='gray')
        if sample_idx == 0:
            ax.set_ylabel(class_names[class_idx], fontsize=12, rotation=0, ha='right', va='center')
        ax.axis('off')

plt.suptitle('Generación controlada por clase - Conditional GAN convolucional', fontsize=16)
plt.tight_layout()
plt.show()